# Mean varaina  var calcution

In [523]:
import pandas as pd
import numpy as np

In [524]:
data = pd.read_csv("All_features.csv",index_col = "Date" )

In [525]:
portfolio = data.loc["2023-01-01":"2025-12-30",["NIFTY50","BANKNIFTY","NASDAQ100","SP500","GOLD"]].dropna()

In [526]:
port_ret = portfolio.pct_change().dropna()

In [527]:
wei  = []
for i in range(len(port_ret.columns)):
    wei.append(1/len(port_ret.columns))
    
wei

[0.2, 0.2, 0.2, 0.2, 0.2]

In [528]:
port_ret["Portfolio_ret"] = port_ret.dot(wei)

In [529]:
port_ret_avg = port_ret["Portfolio_ret"].mean()
port_ret_std = port_ret["Portfolio_ret"].std()

In [530]:
from scipy.stats import norm
import pandas as pd

Mean_variance_var = pd.DataFrame({
    "mean": [port_ret_avg, port_ret_avg, port_ret_avg],
    "std": [port_ret_std, port_ret_std, port_ret_std],
    "CI": [99, 95, 90],
    "alpha": [1, 5, 10],
    "Z_score": [norm.ppf(0.99), norm.ppf(0.95), norm.ppf(0.90)],
    "portfolio_value": [1000000, 1000000, 1000000]
}).T

In [531]:
var =[]
for i in Mean_variance_var:
    var.append((Mean_variance_var.loc["mean"][i] + Mean_variance_var.loc["std"][i] * Mean_variance_var.loc["Z_score"][i])*Mean_variance_var.loc["portfolio_value"][i])
    
    

In [532]:
var

[19469.62848255078, 13930.9565855274, 10978.3110918362]

In [533]:
Mean_variance_var.loc["Var"]  = var

In [534]:
Mean_variance_var

,0,1,2
mean,0.000563,0.000563,0.000563
std,0.008127,0.008127,0.008127
CI,99.000000,95.000000,90.000000
alpha,1.000000,5.000000,10.000000
Z_score,2.326348,1.644854,1.281552
portfolio_value,1000000.000000,1000000.000000,1000000.000000
Var,19469.628483,13930.956586,10978.311092


# Historical var calcution

In [536]:
sort_port_ret = port_ret["Portfolio_ret"].sort_values().to_frame()

In [537]:
sort_port_ret.reset_index( inplace = True)

In [538]:

Hist_var = pd.DataFrame({
    "mean": [port_ret_avg, port_ret_avg, port_ret_avg],
    "std": [port_ret_std, port_ret_std, port_ret_std],
    "CI": [99, 95, 90],
    "alpha": [1, 5, 10],
    "Z_score": [norm.ppf(0.99), norm.ppf(0.95), norm.ppf(0.90)],
    "portfolio_value": [1000000, 1000000, 1000000]
}).T

In [539]:
sort_port_ret["Portfolio_ret"].iloc[round(len(sort_port_ret.index)*0.01)]*1000000

-19436.32792762604

In [540]:
alpha =[0.01,0.05,0.10]
historical_var = []
for i in alpha:
    historical_var.append(sort_port_ret["Portfolio_ret"].iloc[round(len(sort_port_ret.index)*i)]*1000000)

In [541]:
Hist_var.loc["historical_var"] = historical_var

In [542]:
Hist_var

,0,1,2
mean,0.000563,0.000563,0.000563
std,0.008127,0.008127,0.008127
CI,99.000000,95.000000,90.000000
alpha,1.000000,5.000000,10.000000
Z_score,2.326348,1.644854,1.281552
portfolio_value,1000000.000000,1000000.000000,1000000.000000
historical_var,-19436.327928,-11564.465426,-8617.843190


In [543]:
var_no= []
for i in alpha:
    var_no.append(round(len(sort_port_ret.index)*i))

var_no

[7, 34, 68]

In [544]:
Expected_shortfall = []

for i in var_no:
    Expected_shortfall.append(
        sort_port_ret["Portfolio_ret"].iloc[:i].mean() * 1000000
    )
    


# ((sort_port_ret["Portfolio_ret"].iloc[:var_no[0]]).mean())*1000000.000000

In [545]:
Expected_shortfall

[-31371.62909198821, -18157.992905917712, -13974.599452382838]

In [546]:
Hist_var.loc["Espected_shortfall"] = Expected_shortfall

In [547]:
var_of_10_days =[]
for i in Hist_var.loc["historical_var"]:
    var_of_10_days.append(i*np.sqrt(10))

In [548]:
var_of_10_days

[-61463.0656012386, -36570.05066716938, -27252.01299706586]

In [549]:
Hist_var.loc["10_days_var"] = var_of_10_days

In [550]:
Hist_var

,0,1,2
mean,0.000563,0.000563,0.000563
std,0.008127,0.008127,0.008127
CI,99.000000,95.000000,90.000000
alpha,1.000000,5.000000,10.000000
Z_score,2.326348,1.644854,1.281552
portfolio_value,1000000.000000,1000000.000000,1000000.000000
historical_var,-19436.327928,-11564.465426,-8617.843190
Espected_shortfall,-31371.629092,-18157.992906,-13974.599452
10_days_var,-61463.065601,-36570.050667,-27252.012997


# Simulation var calculation

In [552]:
port_ret

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD,Portfolio_ret
Date,,,,,,
2023-01-04,-0.051888,0.007121,-0.011757,-0.006602,0.032157,-0.006194
2023-01-05,0.010920,-0.009715,-0.000165,-0.000247,0.012459,0.002650
2023-01-06,-0.001525,0.016023,-0.007354,-0.018404,-0.002875,-0.002827
2023-01-09,0.013746,0.004560,0.011737,0.014047,0.018898,0.012598
2023-01-10,0.005650,-0.000587,-0.020652,-0.009024,-0.002659,-0.005454
...,...,...,...,...,...,...
2025-12-22,0.026459,0.019076,-0.006121,-0.004538,0.004330,0.007841
2025-12-23,0.004994,0.008595,-0.002463,0.006047,-0.001071,0.003220
2025-12-24,-0.002244,-0.000491,-0.003035,0.000462,0.001738,-0.000714


In [553]:
import pandas as pd
import numpy as np

assets = ["NIFTY50", "BANKNIFTY", "NASDAQ100", "SP500", "GOLD"]

corr_matrix = port_ret[assets].corr()

print(corr_matrix)

            NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
NIFTY50    1.000000   0.129674   0.007840  0.006291  0.150242
BANKNIFTY  0.129674   1.000000   0.034794  0.091066  0.163229
NASDAQ100  0.007840   0.034794   1.000000  0.325186  0.121552
SP500      0.006291   0.091066   0.325186  1.000000  0.125326
GOLD       0.150242   0.163229   0.121552  0.125326  1.000000


In [554]:
L = np.linalg.cholesky(corr_matrix)

print(L)

[[1.         0.         0.         0.         0.        ]
 [0.12967417 0.99155666 0.         0.         0.        ]
 [0.00784002 0.03406508 0.99938887 0.         0.        ]
 [0.00629077 0.09101876 0.3222333  0.94225353 0.        ]
 [0.15024221 0.14497019 0.11550669 0.07849874 0.9679396 ]]


In [555]:
np.random.seed(42)

Z = np.random.randn(10000, len(assets))

Z = pd.DataFrame(Z, columns=assets)

print(Z.head())

    NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
0  0.496714  -0.138264   0.647689  1.523030 -0.234153
1 -0.234137   1.579213   0.767435 -0.469474  0.542560
2 -0.463418  -0.465730   0.241962 -1.913280 -1.724918
3 -0.562288  -1.012831   0.314247 -0.908024 -1.412304
4  1.465649  -0.225776   0.067528 -1.424748 -0.544383


In [556]:
Z_corr = Z.values @ L.T

Z_corr = pd.DataFrame(Z_corr, columns=assets)

print(Z_corr.head())

    NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
0  0.496714  -0.072686   0.646477  1.634327  0.022305
1 -0.234137   1.535517   0.818926 -0.052806  0.770718
2 -0.463418  -0.521891   0.222316 -1.770132 -1.929000
3 -0.562288  -1.077194   0.275145 -0.850052 -1.633315
4  1.465649  -0.033813   0.071287 -1.332044 -0.443499


In [557]:
print("Historical Correlation")
print(corr_matrix)

print("\nCorrelation of Simulated Random Numbers")
print(Z_corr.corr())

Historical Correlation
            NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
NIFTY50    1.000000   0.129674   0.007840  0.006291  0.150242
BANKNIFTY  0.129674   1.000000   0.034794  0.091066  0.163229
NASDAQ100  0.007840   0.034794   1.000000  0.325186  0.121552
SP500      0.006291   0.091066   0.325186  1.000000  0.125326
GOLD       0.150242   0.163229   0.121552  0.125326  1.000000

Correlation of Simulated Random Numbers
            NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
NIFTY50    1.000000   0.140923   0.016058  0.021841  0.157697
BANKNIFTY  0.140923   1.000000   0.024146  0.087412  0.169762
NASDAQ100  0.016058   0.024146   1.000000  0.330757  0.123338
SP500      0.021841   0.087412   0.330757  1.000000  0.126207
GOLD       0.157697   0.169762   0.123338  0.126207  1.000000


In [558]:
import numpy as np
import pandas as pd

assets = ["NIFTY50","BANKNIFTY","NASDAQ100","SP500","GOLD"]

# Annual risk-free rate
rf = 0.065

# Daily time step
dt = 1/252

# Last observed prices
S0 = portfolio.iloc[-1]

# Annualized drift
mu = port_ret[assets].mean() * 252

# Annualized volatility
sigma = port_ret[assets].std() * np.sqrt(252)

# Empty DataFrame for simulated prices
GBM_price = pd.DataFrame(index=Z_corr.index, columns=assets)

# One-day GBM simulation
for asset in assets:

    GBM_price[asset] = (
        S0[asset]
        * np.exp(
            (mu[asset] - 0.5 * sigma[asset]**2) * dt
            + sigma[asset] * np.sqrt(dt) * Z_corr[asset]
        )
    )

GBM_price.head()

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD
0,62.485711,4372.153013,966.448485,440.532869,25873.048050
1,61.617491,4450.754454,968.998882,430.328359,26173.954159
2,61.347609,4350.447063,960.203977,420.183952,25104.675428
3,61.231597,4323.763255,960.979517,425.588848,25219.624309
4,63.655653,4374.036457,957.990271,422.748831,25687.516221


In [559]:
GBM_ret = (GBM_price - S0) / S0

GBM_ret.head()

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD
0,0.009136,0.000470,0.010703,0.023210,0.000714
1,-0.004885,0.018456,0.013370,-0.000492,0.012352
2,-0.009244,-0.004497,0.004173,-0.024054,-0.029005
3,-0.011118,-0.010603,0.004984,-0.011500,-0.024559
4,0.028031,0.000901,0.001857,-0.018096,-0.006462


In [560]:
weights = np.repeat(1/len(assets), len(assets))

GBM_ret["Portfolio_ret"] = GBM_ret.dot(weights)

GBM_ret.head()

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD,Portfolio_ret
0,0.009136,0.000470,0.010703,0.023210,0.000714,0.008847
1,-0.004885,0.018456,0.013370,-0.000492,0.012352,0.007760
2,-0.009244,-0.004497,0.004173,-0.024054,-0.029005,-0.012526
3,-0.011118,-0.010603,0.004984,-0.011500,-0.024559,-0.010559
4,0.028031,0.000901,0.001857,-0.018096,-0.006462,0.001246


In [561]:
sort_port_ret = GBM_ret["Portfolio_ret"].sort_values()

sort_port_ret.head()

1604   -0.031521
927    -0.030242
6361   -0.029347
7923   -0.028116
9947   -0.026407
Name: Portfolio_ret, dtype: float64

In [562]:
GBM_ret_sorted = GBM_ret.sort_values(by="Portfolio_ret")

GBM_ret_sorted.head()

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD,Portfolio_ret
1604,-0.045032,-0.008327,-0.027114,-0.021012,-0.056120,-0.031521
927,-0.054658,0.002840,-0.017789,-0.030737,-0.050866,-0.030242
6361,-0.040042,-0.004284,-0.045586,-0.036207,-0.020617,-0.029347
7923,-0.012197,-0.036801,-0.022736,-0.034234,-0.034612,-0.028116
9947,-0.057584,-0.017758,-0.032715,-0.010443,-0.013535,-0.026407


In [565]:
portfolio_value = 1_000_000

confidence_levels = [0.99, 0.95, 0.90]
alphas = [0.01, 0.05, 0.10]

VaR = []

for alpha in alphas:
    
    index = int(np.ceil(alpha * len(sort_port_ret))) - 1
    
    var = -sort_port_ret.iloc[index] * portfolio_value
    
    VaR.append(var)

VaR

[18266.39330383459, 12748.852825554415, 9927.954233387187]

In [566]:
Expected_shortfall = []

for alpha in alphas:
    
    index = int(np.ceil(alpha * len(sort_port_ret)))
    
    ES = -sort_port_ret.iloc[:index].mean() * portfolio_value
    
    Expected_shortfall.append(ES)

Expected_shortfall

[21249.75993677422, 16163.817714328094, 13741.952534554053]

In [567]:
VaR_10_day = []

for var in VaR:
    VaR_10_day.append(var * np.sqrt(10))

VaR_10_day

[57763.4074765654, 40315.41248302525, 31394.947883414392]

In [568]:
VaR_Table = pd.DataFrame({
    "Confidence Level": [99,95,90],
    "Alpha (%)": [1,5,10],
    "1-Day VaR": np.round(VaR,2),
    "Expected Shortfall": np.round(Expected_shortfall,2),
    "10-Day VaR": np.round(VaR_10_day,2)
})

VaR_Table

,Confidence Level,Alpha (%),1-Day VaR,Expected Shortfall,10-Day VaR
0,99,1,18266.39,21249.76,57763.41
1,95,5,12748.85,16163.82,40315.41
2,90,10,9927.95,13741.95,31394.95


In [614]:
port_ret

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD,Portfolio_ret
Date,,,,,,
2023-01-04,-0.051888,0.007121,-0.011757,-0.006602,0.032157,-0.006194
2023-01-05,0.010920,-0.009715,-0.000165,-0.000247,0.012459,0.002650
2023-01-06,-0.001525,0.016023,-0.007354,-0.018404,-0.002875,-0.002827
2023-01-09,0.013746,0.004560,0.011737,0.014047,0.018898,0.012598
2023-01-10,0.005650,-0.000587,-0.020652,-0.009024,-0.002659,-0.005454
...,...,...,...,...,...,...
2025-12-22,0.026459,0.019076,-0.006121,-0.004538,0.004330,0.007841
2025-12-23,0.004994,0.008595,-0.002463,0.006047,-0.001071,0.003220
2025-12-24,-0.002244,-0.000491,-0.003035,0.000462,0.001738,-0.000714


# Backtest monte carlo var 

In [620]:
backtest = data.loc["2020 - 01-01":"2023-12-30",["NIFTY50","BANKNIFTY","NASDAQ100","SP500","GOLD"]].dropna()

In [622]:
backtest

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD
Date,,,,,
2020-01-02,66.250000,1524.500000,307.534424,332.919128,28543.519531
2020-01-03,68.599998,1549.199951,302.458679,330.041199,28451.500000
2020-01-06,68.910004,1566.199951,289.134949,329.134949,28226.189453
2020-01-07,68.269997,1571.800049,288.591095,332.779724,28322.060547
2020-01-08,65.440002,1557.400024,289.860046,329.593048,28087.919922
...,...,...,...,...,...
2023-12-21,79.389999,2039.099976,609.680237,369.708069,16621.130859
2023-12-22,79.070000,2057.100098,603.050720,371.264557,16340.410156
2023-12-27,79.650002,2081.899902,614.226257,379.615753,16624.839844


In [624]:
assets = ["NIFTY50","BANKNIFTY","NASDAQ100","SP500","GOLD"]

backtest_ret = backtest[assets].pct_change().dropna()

In [626]:
weights = np.ones(len(assets)) / len(assets)

backtest_ret["Portfolio_ret"] = backtest_ret.dot(weights)

In [628]:
VaR_99 = VaR_Table.loc[VaR_Table["Confidence Level"]==99,"1-Day VaR"].iloc[0] / 1000000

VaR_95 = VaR_Table.loc[VaR_Table["Confidence Level"]==95,"1-Day VaR"].iloc[0] / 1000000

VaR_90 = VaR_Table.loc[VaR_Table["Confidence Level"]==90,"1-Day VaR"].iloc[0] / 1000000

In [630]:
breach_99 = (backtest_ret["Portfolio_ret"] < -VaR_99).sum()

breach_95 = (backtest_ret["Portfolio_ret"] < -VaR_95).sum()

breach_90 = (backtest_ret["Portfolio_ret"] < -VaR_90).sum()

In [632]:
n = len(backtest_ret)

expected_99 = n * 0.01
expected_95 = n * 0.05
expected_90 = n * 0.10

In [634]:
Backtest = pd.DataFrame({

    "Confidence":[99,95,90],

    "Expected Breaches":[expected_99,
                         expected_95,
                         expected_90],

    "Actual Breaches":[breach_99,
                       breach_95,
                       breach_90]

})

Backtest

,Confidence,Expected Breaches,Actual Breaches
0,99,9.16,47
1,95,45.80,91
2,90,91.60,130


# monte carlo using t distbuion 

In [648]:
import pandas as pd
import numpy as np

assets = ["NIFTY50", "BANKNIFTY", "NASDAQ100", "SP500", "GOLD"]

corr_matrix = port_ret[assets].corr()

print(corr_matrix)

            NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
NIFTY50    1.000000   0.129674   0.007840  0.006291  0.150242
BANKNIFTY  0.129674   1.000000   0.034794  0.091066  0.163229
NASDAQ100  0.007840   0.034794   1.000000  0.325186  0.121552
SP500      0.006291   0.091066   0.325186  1.000000  0.125326
GOLD       0.150242   0.163229   0.121552  0.125326  1.000000


In [650]:
L = np.linalg.cholesky(corr_matrix)

print(L)

[[1.         0.         0.         0.         0.        ]
 [0.12967417 0.99155666 0.         0.         0.        ]
 [0.00784002 0.03406508 0.99938887 0.         0.        ]
 [0.00629077 0.09101876 0.3222333  0.94225353 0.        ]
 [0.15024221 0.14497019 0.11550669 0.07849874 0.9679396 ]]


In [652]:
np.random.seed(42)

from scipy.stats import t

df = 6          # Example degrees of freedom

Z = t.rvs(df=df, size=(10000, 5))

Z = pd.DataFrame(Z, columns=assets)

print(Z.head())

    NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
0  0.549962  -1.072880   1.346520 -0.730348  0.539403
1  1.514367  -1.604038  -0.561772 -0.558335 -0.488354
2 -0.394496  -0.566979   1.341074  2.124760  0.743890
3 -1.922218  -0.364462  -1.151416 -0.601822  0.480950
4 -0.981574  -1.180599  -1.785791  0.810716  0.474058


In [654]:
from scipy.stats import t

df = 6          # Example degrees of freedom

Z = t.rvs(df=df, size=(10000, 5))

In [658]:
Z_corr = Z @ L.T
Z_corr = pd.DataFrame(Z_corr, columns=assets)

print(Z_corr.head())

    NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
0  0.205909   0.332495   1.117521 -0.008831 -1.827840
1 -0.267096  -0.724170   0.474500  0.373674 -1.872617
2  0.206649   0.216160  -1.204575 -0.944670  0.893936
3 -0.237061  -1.422917  -0.039902 -0.043493 -2.992430
4 -0.465177  -0.457769  -0.541019  0.034217  0.852577


In [660]:
print("Historical Correlation")
print(corr_matrix)

print("\nCorrelation of Simulated Random Numbers")
print(Z_corr.corr())

Historical Correlation
            NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
NIFTY50    1.000000   0.129674   0.007840  0.006291  0.150242
BANKNIFTY  0.129674   1.000000   0.034794  0.091066  0.163229
NASDAQ100  0.007840   0.034794   1.000000  0.325186  0.121552
SP500      0.006291   0.091066   0.325186  1.000000  0.125326
GOLD       0.150242   0.163229   0.121552  0.125326  1.000000

Correlation of Simulated Random Numbers
            NIFTY50  BANKNIFTY  NASDAQ100     SP500      GOLD
NIFTY50    1.000000   0.128523  -0.006545  0.005796  0.129514
BANKNIFTY  0.128523   1.000000   0.033773  0.103577  0.172388
NASDAQ100 -0.006545   0.033773   1.000000  0.337191  0.107284
SP500      0.005796   0.103577   0.337191  1.000000  0.134447
GOLD       0.129514   0.172388   0.107284  0.134447  1.000000


In [662]:
import numpy as np
import pandas as pd

assets = ["NIFTY50","BANKNIFTY","NASDAQ100","SP500","GOLD"]

# Annual risk-free rate
rf = 0.065

# Daily time step
dt = 1/252

# Last observed prices
S0 = portfolio.iloc[-1]

# Annualized drift
mu = port_ret[assets].mean() * 252

# Annualized volatility
sigma = port_ret[assets].std() * np.sqrt(252)

# Empty DataFrame for simulated prices
GBM_price = pd.DataFrame(index=Z_corr.index, columns=assets)

# One-day GBM simulation
for asset in assets:

    GBM_price[asset] = (
        S0[asset]
        * np.exp(
            (mu[asset] - 0.5 * sigma[asset]**2) * dt
            + sigma[asset] * np.sqrt(dt) * Z_corr[asset]
        )
    )

GBM_price.head()

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD
0,62.138791,4391.824562,973.430812,430.591315,25143.942774
1,61.578623,4340.707961,963.911749,432.885353,25126.553924
2,62.139671,4386.167446,939.491907,425.029831,26223.829781
3,61.614042,4307.233053,956.363792,430.384033,24695.575238
4,61.345543,4353.538861,949.067612,430.848878,26207.078013


In [664]:
GBM_ret = (GBM_price - S0) / S0

GBM_ret.head()

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD
0,0.003533,0.004971,0.018005,0.000119,-0.027487
1,-0.005513,-0.006726,0.008050,0.005447,-0.028159
2,0.003548,0.003677,-0.017488,-0.012798,0.014281
3,-0.004941,-0.014386,0.000156,-0.000362,-0.044829
4,-0.009277,-0.003790,-0.007474,0.000717,0.013633


In [666]:
weights = np.repeat(1/len(assets), len(assets))

GBM_ret["Portfolio_ret"] = GBM_ret.dot(weights)

GBM_ret.head()

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD,Portfolio_ret
0,0.003533,0.004971,0.018005,0.000119,-0.027487,-0.000172
1,-0.005513,-0.006726,0.008050,0.005447,-0.028159,-0.005380
2,0.003548,0.003677,-0.017488,-0.012798,0.014281,-0.001756
3,-0.004941,-0.014386,0.000156,-0.000362,-0.044829,-0.012872
4,-0.009277,-0.003790,-0.007474,0.000717,0.013633,-0.001238


In [668]:
sort_port_ret = GBM_ret["Portfolio_ret"].sort_values()

sort_port_ret.head()

2761   -0.044816
6937   -0.040472
759    -0.036779
5000   -0.036267
3806   -0.035197
Name: Portfolio_ret, dtype: float64

In [670]:
GBM_ret_sorted = GBM_ret.sort_values(by="Portfolio_ret")

GBM_ret_sorted.head()

,NIFTY50,BANKNIFTY,NASDAQ100,SP500,GOLD,Portfolio_ret
2761,-0.025771,-0.008788,-0.143554,-0.045419,-0.000547,-0.044816
6937,-0.162738,-0.056222,0.020422,0.010792,-0.014614,-0.040472
759,-0.049371,-0.031302,-0.032782,-0.021967,-0.048472,-0.036779
5000,-0.067658,-0.026298,-0.032404,-0.020075,-0.034899,-0.036267
3806,-0.046623,-0.030543,-0.014648,-0.002450,-0.081721,-0.035197


In [672]:
portfolio_value = 1_000_000

confidence_levels = [0.99, 0.95, 0.90]
alphas = [0.01, 0.05, 0.10]

VaR = []

for alpha in alphas:
    
    index = int(np.ceil(alpha * len(sort_port_ret))) - 1
    
    var = -sort_port_ret.iloc[index] * portfolio_value
    
    VaR.append(var)

VaR

[23382.216149672022, 15558.209311619552, 11675.8182242072]

In [676]:
Expected_shortfall = []

for alpha in alphas:
    
    index = int(np.ceil(alpha * len(sort_port_ret)))
    
    ES = -sort_port_ret.iloc[:index].mean() * portfolio_value
    
    Expected_shortfall.append(ES)

Expected_shortfall

[27377.77469458855, 20262.99868486576, 16827.62196374914]

In [678]:
VaR_10_day = []

for var in VaR:
    VaR_10_day.append(var * np.sqrt(10))

VaR_10_day

[73941.05977533614, 49199.37773835817, 36922.17913459727]

In [680]:
T_VaR_Table = pd.DataFrame({
    "Confidence Level": [99,95,90],
    "Alpha (%)": [1,5,10],
    "1-Day VaR": np.round(VaR,2),
    "Expected Shortfall": np.round(Expected_shortfall,2),
    "10-Day VaR": np.round(VaR_10_day,2)
})

T_VaR_Table

,Confidence Level,Alpha (%),1-Day VaR,Expected Shortfall,10-Day VaR
0,99,1,23382.22,27377.77,73941.06
1,95,5,15558.21,20263.00,49199.38
2,90,10,11675.82,16827.62,36922.18


# Backtest of var usnign Tdisbtuion 

In [684]:
VaR_99 = T_VaR_Table.loc[T_VaR_Table["Confidence Level"]==99,"1-Day VaR"].iloc[0] / 1000000

VaR_95 = T_VaR_Table.loc[T_VaR_Table["Confidence Level"]==95,"1-Day VaR"].iloc[0] / 1000000

VaR_90 = T_VaR_Table.loc[T_VaR_Table["Confidence Level"]==90,"1-Day VaR"].iloc[0] / 1000000

In [686]:
breach_99 = (backtest_ret["Portfolio_ret"] < -VaR_99).sum()

breach_95 = (backtest_ret["Portfolio_ret"] < -VaR_95).sum()

breach_90 = (backtest_ret["Portfolio_ret"] < -VaR_90).sum()

In [688]:
n = len(backtest_ret)

expected_99 = n * 0.01
expected_95 = n * 0.05
expected_90 = n * 0.10

In [690]:
Backtest = pd.DataFrame({

    "Confidence":[99,95,90],

    "Expected Breaches":[expected_99,
                         expected_95,
                         expected_90],

    "Actual Breaches":[breach_99,
                       breach_95,
                       breach_90]

})

Backtest

,Confidence,Expected Breaches,Actual Breaches
0,99,9.16,23
1,95,45.80,58
2,90,91.60,102
